In [3]:
from pystac_client import Client
import requests
import os
from tqdm import tqdm

In [4]:
from utils.constants import GHANA, SENTINEL_SCENES_FOLDERPATH

In [5]:
api_url = "https://earth-search.aws.element84.com/v1"
client = Client.open(api_url)

In [6]:
ghana_bbox = list(GHANA.bounds.iloc[0])

print("Searching for satellite imagery...")
search = client.search(
    collections=["sentinel-2-l2a"],
    bbox=ghana_bbox,
    datetime="2025-01-10/2025-01-17",
    query={"eo:cloud_cover": {"lt": 20}} 
)

items = list(search.items())
print(f"Found {len(items)} scenes.\n")

Searching for satellite imagery...
Found 98 scenes.



In [7]:
BANDS = ["blue", "green", "red", "nir", "swir22"]

In [8]:
scenes = list(search.items())
print(f"Found {len(scenes)} cloud-free scenes for Ghana in Jan 2025.\n")

scene = scenes[0]
print(f"Direct download links for Scene ID: {scene.id}")

Found 98 cloud-free scenes for Ghana in Jan 2025.

Direct download links for Scene ID: S2B_30NYM_20250117_0_L2A


In [9]:
# E.G.
for band in BANDS:
    if band in scene.assets:
        download_url = scene.assets[band].href
        print(f"{band.upper()} (B{BANDS.index(band)+1:02d}): {download_url}")

BLUE (B01): https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/N/YM/2025/1/S2B_30NYM_20250117_0_L2A/B02.tif
GREEN (B02): https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/N/YM/2025/1/S2B_30NYM_20250117_0_L2A/B03.tif
RED (B03): https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/N/YM/2025/1/S2B_30NYM_20250117_0_L2A/B04.tif
NIR (B04): https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/N/YM/2025/1/S2B_30NYM_20250117_0_L2A/B08.tif
SWIR22 (B05): https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/N/YM/2025/1/S2B_30NYM_20250117_0_L2A/B12.tif


In [10]:
def download_file(url: str, filepath: Path) -> None:
    """Downloads a file using a .part extension to prevent corruption if interrupted."""
    if filepath.exists():
        return

    part_filepath = filepath.with_suffix(filepath.suffix + ".part")
    
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    
    with part_filepath.open('wb') as f, tqdm(desc=filepath.name, total=total_size, unit='iB', unit_scale=True, unit_divisor=1024, leave=False) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk: 
                size = f.write(chunk)
                bar.update(size)
                
    part_filepath.rename(filepath)

In [11]:
for scene in scenes: 
    scene_folder = SENTINEL_SCENES_FOLDERPATH / scene.id
    scene_folder.mkdir(parents=True, exist_ok=True)
    
    print(f"Downloading Scene: {scene.id}")
    
    for band in BANDS:
        if band in scene.assets:
            download_url = scene.assets[band].href
            filename = f"{scene.id}_{band}.tif"
            filepath = scene_folder / filename
            
            try:
                download_file(download_url, filepath)
            except Exception as e:
                print(f"Error downloading {band}: {e}")

KeyboardInterrupt: 